# 🧪 Building a Unified Arabic AI-Detection Corpus

**Goal.** Combine the two KFUPM-JRCAI "Arabic AI Fingerprint" datasets into one
balanced four-bucket corpus suitable for binary AI-vs-Human detection.

**Buckets:**
1. Human-written research abstracts
2. Human-written social media posts
3. AI-generated research abstracts (4 LLMs × 3 generation methods)
4. AI-generated social media posts (4 LLMs × 1 generation method)

**Design choices (locked in by user):**
- **Granularity:** abstracts only (no full papers)
- **AI side:** use KFUPM's existing generations (ALLaM, Jais, Llama 3.1, GPT-4)
- **Balance:** 50/50 Human vs AI, natural sizes (no forced downsampling)

**Output.** A single Parquet/CSV file with columns
`text · label · domain · source · llm` plus train/val/test splits stratified
on both `label` and `domain`.

> Run all cells top to bottom. Total runtime: ~3-5 minutes on Colab free tier.


## 1. Install dependencies

In [ ]:
!pip install -q datasets pandas pyarrow scikit-learn

## 2. Imports & configuration

In [ ]:
import re
import unicodedata
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split

# ---------- Config ----------
SEED          = 42
MIN_WORDS_ABS = 30   # KFUPM threshold for abstracts
MIN_WORDS_SM  = 50   # KFUPM threshold for social-media posts
TEST_SIZE     = 0.10
VAL_SIZE      = 0.10  # of the remaining 90% post-test split

LLMS_ABS = ["allam", "jais", "llama", "openai"]
LLMS_SM  = ["allam", "jais", "llama", "openai"]

np.random.seed(SEED)
print('Config loaded.')

Config loaded.


## 3. Load both datasets from Hugging Face

These are public datasets — no auth token required.

In [ ]:
abstracts = load_dataset("KFUPM-JRCAI/arabic-generated-abstracts")
social    = load_dataset("KFUPM-JRCAI/arabic-generated-social-media-posts")

print('Abstracts splits :', {k: len(v) for k, v in abstracts.items()})
print('Social   splits :', {k: len(v) for k, v in social.items()})

README.md:   0%|          | 0.00/8.10k [00:00<?, ?B/s]

data/by_polishing-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 8.49MB            

data/by_polishing-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/from_title-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.90MB            

data/from_title-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

README.md:   0%|          | 0.00/7.77k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 43.4MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3318 [00:00<?, ? examples/s]

Abstracts splits : {'by_polishing': 2851, 'from_title': 2963, 'from_title_and_content': 2574}
Social   splits : {'train': 3318}


## 4. Preprocessing pipeline

Lightweight Arabic-aware cleaning that **preserves diacritics** (matching the
strategy used in your Aero-BiLSTM-X / Aero-CamelBERT-X work).

Steps:
- Unicode NFKC normalization
- Remove tatweel (ـ) and zero-width chars
- Normalize Alef variants (آأإ → ا), ى → ي, ة → ه
- Collapse repeated whitespace
- Strip control / non-printable characters

In [ ]:
# Compiled regexes once for speed
_TATWEEL  = re.compile('\u0640')
_ZWJ_ZWNJ = re.compile('[\u200B-\u200F\u202A-\u202E\uFEFF]')
_ALEF     = re.compile('[\u0622\u0623\u0625]')   # آ أ إ
_YA       = re.compile('\u0649')                  # ى
_TAA_M    = re.compile('\u0629')                  # ة
_WHITES   = re.compile(r'\s+')
_CTRL     = re.compile(r'[\x00-\x1F\x7F]')

def normalize_ar(text: str) -> str:
    if not isinstance(text, str) or not text:
        return ''
    t = unicodedata.normalize('NFKC', text)
    t = _CTRL.sub(' ', t)
    t = _ZWJ_ZWNJ.sub('', t)
    t = _TATWEEL.sub('', t)
    t = _ALEF.sub('\u0627', t)   # → ا
    t = _YA.sub('\u064A', t)     # → ي
    t = _TAA_M.sub('\u0647', t)  # → ه
    t = _WHITES.sub(' ', t).strip()
    return t

def word_count(text: str) -> int:
    return len(text.split()) if text else 0

# Smoke test
sample = "هـٰذا   اختبارٌ للأَلِف وللتَّاء المربوطة"
print('Before:', sample)
print('After :', normalize_ar(sample))

Before: هـٰذا   اختبارٌ للأَلِف وللتَّاء المربوطة
After : هٰذا اختبارٌ للاَلِف وللتَّاء المربوطه


## 5. Melt the abstracts dataset

Each row currently has 1 human + 4 LLM-generated columns side by side.
We "melt" it into long format: one (text, label, llm) tuple per row.

In [ ]:
def melt_abstracts(ds_split, split_name):
    df = ds_split.to_pandas()
    rows = []
    for _, r in df.iterrows():
        # human row
        rows.append({
            'text'  : r['original_abstract'],
            'label' : 'human',
            'llm'   : 'human',
            'method': split_name,
        })
        # 4 LLM rows
        for llm in LLMS_ABS:
            col = f'{llm}_generated_abstract'
            if col in r and isinstance(r[col], str):
                rows.append({
                    'text'  : r[col],
                    'label' : 'ai',
                    'llm'   : llm,
                    'method': split_name,
                })
    return pd.DataFrame(rows)

abs_frames = []
for split_name in ['by_polishing', 'from_title', 'from_title_and_content']:
    abs_frames.append(melt_abstracts(abstracts[split_name], split_name))
abs_long = pd.concat(abs_frames, ignore_index=True)
abs_long['domain'] = 'research'
abs_long['source'] = 'ASJP'

print('Abstracts long-format size :', len(abs_long))
print(abs_long['label'].value_counts().to_string())
print()
print('AI breakdown by LLM:')
print(abs_long[abs_long['label']=='ai']['llm'].value_counts().to_string())

Abstracts long-format size : 41940
label
ai       33552
human     8388

AI breakdown by LLM:
llm
allam     8388
jais      8388
llama     8388
openai    8388


## 6. Melt the social-media dataset

In [ ]:
def melt_social(ds_split):
    df = ds_split.to_pandas()
    rows = []
    for _, r in df.iterrows():
        rows.append({
            'text'  : r['original_post'],
            'label' : 'human',
            'llm'   : 'human',
            'method': 'by_polishing',
        })
        for llm in LLMS_SM:
            col = f'{llm}_generated_post'
            if col in r and isinstance(r[col], str):
                rows.append({
                    'text'  : r[col],
                    'label' : 'ai',
                    'llm'   : llm,
                    'method': 'by_polishing',
                })
    return pd.DataFrame(rows)

sm_long = melt_social(social['train'])
sm_long['domain'] = 'social_media'
sm_long['source'] = 'BRAD+HARD'

print('Social long-format size:', len(sm_long))
print(sm_long['label'].value_counts().to_string())

Social long-format size: 16590
label
ai       13272
human     3318


## 7. Clean, filter, deduplicate

- Apply Arabic normalization
- Drop rows below the minimum word threshold
- Drop exact duplicates (same `text`)

In [ ]:
def clean_and_filter(df, min_words):
    df = df.copy()
    df['text'] = df['text'].astype(str).map(normalize_ar)
    df['n_words'] = df['text'].map(word_count)
    n0 = len(df)
    df = df[df['n_words'] >= min_words].copy()
    n1 = len(df)
    df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
    n2 = len(df)
    print(f'  raw          : {n0:>6}')
    print(f'  after length : {n1:>6}  (-{n0-n1})')
    print(f'  after dedupe : {n2:>6}  (-{n1-n2})')
    return df

print('Abstracts:')
abs_clean = clean_and_filter(abs_long, MIN_WORDS_ABS)
print()
print('Social:')
sm_clean  = clean_and_filter(sm_long, MIN_WORDS_SM)

Abstracts:
  raw          :  41940
  after length :  41940  (-0)
  after dedupe :  36525  (-5415)

Social:
  raw          :  16590
  after length :  16590  (-0)
  after dedupe :  16529  (-61)


## 8. Balance to 50/50 per domain

The constraint is always the **human** count (smaller pool).
We sample an equal number of AI rows, evenly across the 4 LLMs.

In [ ]:
def balance_5050(df, seed=SEED):
    humans = df[df['label']=='human'].copy()
    ai     = df[df['label']=='ai'].copy()
    n_h    = len(humans)
    # Sample n_h AI rows, stratified by llm so each LLM contributes ~equally
    per_llm = n_h // len(LLMS_ABS)
    leftover = n_h - per_llm * len(LLMS_ABS)
    ai_balanced = []
    for i, llm in enumerate(LLMS_ABS):
        take = per_llm + (1 if i < leftover else 0)
        pool = ai[ai['llm']==llm]
        if len(pool) >= take:
            ai_balanced.append(pool.sample(n=take, random_state=seed))
        else:
            ai_balanced.append(pool)  # take all if pool is smaller
    ai_balanced = pd.concat(ai_balanced, ignore_index=True)
    out = pd.concat([humans, ai_balanced], ignore_index=True)
    return out.sample(frac=1, random_state=seed).reset_index(drop=True)

abs_bal = balance_5050(abs_clean)
sm_bal  = balance_5050(sm_clean)

print('Research domain balanced:')
print(abs_bal['label'].value_counts().to_string())
print()
print('Social-media domain balanced:')
print(sm_bal['label'].value_counts().to_string())

Research domain balanced:
label
human    2992
ai       2992

Social-media domain balanced:
label
human    3318
ai       3318


## 9. Combine and create train/val/test splits

Stratified on a combined `(label, domain)` key so every split keeps both
the 50/50 label balance and the research/social-media domain mix.

In [ ]:
corpus = pd.concat([abs_bal, sm_bal], ignore_index=True)
corpus = corpus.sample(frac=1, random_state=SEED).reset_index(drop=True)
print('Total corpus size:', len(corpus))
print()
print('Cross-tab — label × domain:')
print(pd.crosstab(corpus['label'], corpus['domain']))

# Stratification key
strata = corpus['label'].astype(str) + '|' + corpus['domain'].astype(str)

# 1) hold out the test set
train_val, test = train_test_split(
    corpus, test_size=TEST_SIZE, stratify=strata, random_state=SEED)
# 2) split val from the remainder
strata_tv = train_val['label'].astype(str) + '|' + train_val['domain'].astype(str)
train, val = train_test_split(
    train_val, test_size=VAL_SIZE/(1-TEST_SIZE), stratify=strata_tv, random_state=SEED)

train['split'] = 'train'
val  ['split'] = 'val'
test ['split'] = 'test'

print()
print('Split sizes:')
print(f'  train: {len(train):>5}')
print(f'  val  : {len(val):>5}')
print(f'  test : {len(test):>5}')

Total corpus size: 12620

Cross-tab — label × domain:
domain  research  social_media
label                         
ai          2992          3318
human       2992          3318

Split sizes:
  train: 10096
  val  :  1262
  test :  1262


## 10. Integrity checks

Confirm every split is balanced on both axes and there's no leakage between
train and test.

In [ ]:
for name, df in [('train', train), ('val', val), ('test', test)]:
    print(f'== {name.upper()} ==')
    print(pd.crosstab(df['label'], df['domain']))
    print()

# Leakage check
leak = set(train['text']).intersection(set(test['text']))
print(f'Train↔Test text overlap: {len(leak)} rows')

== TRAIN ==
domain  research  social_media
label                         
ai          2394          2654
human       2394          2654

== VAL ==
domain  research  social_media
label                         
ai           299           332
human        299           332

== TEST ==
domain  research  social_media
label                         
ai           299           332
human        299           332

Train↔Test text overlap: 0 rows


## 11. Save the final corpus

Saves both Parquet (compact, preserves dtypes) and CSV (universally readable).
Also exports a single combined file with a `split` column.

In [ ]:
final = pd.concat([train, val, test], ignore_index=True)

# Single combined file with split column
final.to_parquet('arabic_ai_detection_corpus.parquet', index=False)
final.to_csv    ('arabic_ai_detection_corpus.csv',     index=False)

# Per-split files
train.to_parquet('train.parquet', index=False)
val  .to_parquet('val.parquet',   index=False)
test .to_parquet('test.parquet',  index=False)

print('Files written:')
print('  arabic_ai_detection_corpus.parquet  (combined, with split col)')
print('  arabic_ai_detection_corpus.csv      (combined, with split col)')
print('  train.parquet · val.parquet · test.parquet')

Files written:
  arabic_ai_detection_corpus.parquet  (combined, with split col)
  arabic_ai_detection_corpus.csv      (combined, with split col)
  train.parquet · val.parquet · test.parquet


## 12. Quick sanity peek

In [ ]:
print('Schema:')
print(final.dtypes)
print()
print('Sample row (HUMAN, research):')
print(final[(final.label=='human') & (final.domain=='research')].iloc[0])
print()
print('Sample row (AI, social_media):')
print(final[(final.label=='ai') & (final.domain=='social_media')].iloc[0])

Schema:
text       object
label      object
llm        object
method     object
domain     object
source     object
n_words     int64
split      object
dtype: object

Sample row (HUMAN, research):
text       شك ان الدراسات القرانيه التي تجعل من القران ال...
label                                                  human
llm                                                    human
method                                          by_polishing
domain                                              research
source                                                  ASJP
n_words                                                  185
split                                                  train
Name: 6, dtype: object

Sample row (AI, social_media):
text       المراجعه بعد القراءه الثانيه :) ميلاد مُجتمع ي...
label                                                     ai
llm                                                    allam
method                                          by_polishing
domain          

## 13. (Optional) Download from Colab to your machine

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Uncomment if you're running on Google Colab
from google.colab import files
# files.download('arabic_ai_detection_corpus.parquet')
files.download('arabic_ai_detection_corpus.csv')

In [ ]:
"""
DIAGNOSTIC ONLY — measures source-family leakage in your existing saved
train/val/test split. Does not change your pipeline, does not retrain
anything, does not touch your saved corpus files.

HOW TO USE
----------
Paste this as a new cell in build_arabic_ai_corpus.ipynb, right after
cell 6 (where `abstracts` and `social` are loaded from Hugging Face) and
cell 8 (where `normalize_ar` is defined). It reuses your own LLMS_ABS,
LLMS_SM, and normalize_ar exactly as already defined, so the re-melted
text matches your saved corpus's normalization exactly.

It will look for `train`, `val`, `test` already in memory (if you're
running this in the same session after cell 18). If they are not in
memory, it falls back to loading train.parquet / val.parquet /
test.parquet from disk (uncomment the fallback path below and adjust
the folder if needed).

WHAT IT DOES
------------
1. Re-melts the same wide-format HF datasets you already loaded, but
   keeps a `family_id` marking which human original each AI row came
   from (family_id = "abs__<method>__<row_index>" or "sm__<row_index>").
2. Applies your existing normalize_ar() so re-melted text matches the
   normalization already baked into your saved corpus.
3. Matches each re-melted row against your saved train/val/test split
   by exact normalized text.
4. For every family_id with at least one matched row, reports which
   split(s) it landed in. A family that appears in more than one split
   is leakage: a human original and at least one of its own AI
   paraphrases ended up on different sides of your train/test boundary.

It prints a summary and returns a DataFrame of the leaked families so
you can inspect specific cases directly.
"""

import pandas as pd

# ---------------------------------------------------------------------
# 0. Load the saved split (in-memory if available, else from disk)
# ---------------------------------------------------------------------
import os

try:
    train, val, test  # noqa: F821 (expected to already exist in the notebook)
    print("Using train/val/test already in memory.")
except NameError:
    print("train/val/test not in memory, looking for saved files instead.")

    per_split_files = ['train.parquet', 'val.parquet', 'test.parquet']
    combined_candidates = [
        'arabic_ai_detection_corpus.parquet',
        'arabic_ai_detection_corpus.csv',
    ]

    if all(os.path.exists(p) for p in per_split_files):
        print("Found train.parquet / val.parquet / test.parquet.")
        train = pd.read_parquet('train.parquet')
        val = pd.read_parquet('val.parquet')
        test = pd.read_parquet('test.parquet')

    else:
        combined_path = next((p for p in combined_candidates if os.path.exists(p)), None)
        if combined_path is None:
            raise FileNotFoundError(
                "Could not find train/val/test parquet files, nor a combined "
                "arabic_ai_detection_corpus.parquet/.csv with a 'split' column, "
                "in the current working directory "
                f"({os.getcwd()}).\n\n"
                "This usually means one of two things:\n"
                "  1. You're in a fresh Colab session that never re-ran the "
                "build pipeline (cells 1-22 of build_arabic_ai_corpus.ipynb) "
                "in this session, so the files were never written here.\n"
                "  2. You have the corpus file saved locally on your own "
                "machine but haven't uploaded it into this Colab session's "
                "working directory yet (use the file-upload icon in the left "
                "sidebar, or google.colab.files.upload()).\n\n"
                "Either re-run the build pipeline through cell 22 in this "
                "session first, or upload one of: "
                f"{per_split_files + combined_candidates}, then re-run this cell."
            )
        print(f"Found combined file: {combined_path}")
        if combined_path.endswith('.parquet'):
            combined = pd.read_parquet(combined_path)
        else:
            combined = pd.read_csv(combined_path)
        if 'split' not in combined.columns:
            raise ValueError(
                f"{combined_path} does not have a 'split' column, so it can't "
                "be divided into train/val/test automatically. Check that "
                "this is the final combined corpus file, not an intermediate one."
            )
        train = combined[combined['split'] == 'train'].reset_index(drop=True)
        val = combined[combined['split'] == 'val'].reset_index(drop=True)
        test = combined[combined['split'] == 'test'].reset_index(drop=True)

split_lookup = {}
for name, df in [('train', train), ('val', val), ('test', test)]:
    for t in df['text']:
        split_lookup[t] = name

print(f"Split lookup built: {len(split_lookup)} unique texts across "
      f"train={len(train)}, val={len(val)}, test={len(test)}")

# ---------------------------------------------------------------------
# 1. Re-melt with family_id, reusing your existing LLMS_ABS / LLMS_SM /
#    normalize_ar exactly as already defined in your notebook.
# ---------------------------------------------------------------------

def melt_abstracts_with_family(ds_split, split_name):
    df = ds_split.to_pandas()
    rows = []
    for idx, r in df.iterrows():
        family_id = f"abs__{split_name}__{idx}"
        rows.append({
            'text': normalize_ar(r['original_abstract']),  # noqa: F821
            'label': 'human',
            'llm': 'human',
            'family_id': family_id,
        })
        for llm in LLMS_ABS:  # noqa: F821
            col = f'{llm}_generated_abstract'
            if col in r and isinstance(r[col], str):
                rows.append({
                    'text': normalize_ar(r[col]),  # noqa: F821
                    'label': 'ai',
                    'llm': llm,
                    'family_id': family_id,
                })
    return pd.DataFrame(rows)


def melt_social_with_family(ds_split):
    df = ds_split.to_pandas()
    rows = []
    for idx, r in df.iterrows():
        family_id = f"sm__{idx}"
        rows.append({
            'text': normalize_ar(r['original_post']),  # noqa: F821
            'label': 'human',
            'llm': 'human',
            'family_id': family_id,
        })
        for llm in LLMS_SM:  # noqa: F821
            col = f'{llm}_generated_post'
            if col in r and isinstance(r[col], str):
                rows.append({
                    'text': normalize_ar(r[col]),  # noqa: F821
                    'label': 'ai',
                    'llm': llm,
                    'family_id': family_id,
                })
    return pd.DataFrame(rows)


print("\nRe-melting abstracts with family_id...")
abs_frames = []
for split_name in ['by_polishing', 'from_title', 'from_title_and_content']:
    abs_frames.append(melt_abstracts_with_family(abstracts[split_name], split_name))  # noqa: F821
abs_long_fam = pd.concat(abs_frames, ignore_index=True)

print("Re-melting social media with family_id...")
sm_long_fam = melt_social_with_family(social['train'])  # noqa: F821

long_fam = pd.concat([abs_long_fam, sm_long_fam], ignore_index=True)
print(f"Total re-melted rows: {len(long_fam)}")

# ---------------------------------------------------------------------
# 2. Match against the saved split and measure leakage
# ---------------------------------------------------------------------
long_fam['split'] = long_fam['text'].map(split_lookup)
matched = long_fam.dropna(subset=['split']).copy()

print(f"\nMatched {len(matched)} of {len(long_fam)} re-melted rows "
      f"against your saved corpus ({100 * len(matched) / len(long_fam):.1f}%). "
      f"The rest were dropped by your existing length filter, dedup, or "
      f"50/50 balancing step, which is expected.")

fam_report = matched.groupby('family_id').agg(
    n_splits=('split', 'nunique'),
    splits=('split', lambda s: sorted(s.unique().tolist())),
    n_rows=('split', 'size'),
    labels=('label', lambda s: sorted(s.unique().tolist())),
).reset_index()

leaked = fam_report[fam_report['n_splits'] > 1].sort_values('n_rows', ascending=False)

n_families = len(fam_report)
n_leaked = len(leaked)
rows_in_leaked_families = matched[matched['family_id'].isin(leaked['family_id'])]

print("\n" + "=" * 60)
print("FAMILY LEAKAGE REPORT")
print("=" * 60)
print(f"Families with at least one row in the saved corpus : {n_families}")
print(f"Families split across more than one partition       : {n_leaked} "
      f"({100 * n_leaked / max(n_families, 1):.2f}%)")
print(f"Total rows belonging to a leaked family              : {len(rows_in_leaked_families)} "
      f"out of {len(matched)} matched rows "
      f"({100 * len(rows_in_leaked_families) / max(len(matched), 1):.2f}%)")

if n_leaked:
    print("\nBreakdown by which splits are involved:")
    print(leaked['splits'].astype(str).value_counts().to_string())

    print("\nHow many leaked families involve the test set specifically "
          "(the number that matters most, since this is what inflates "
          "reported accuracy):")
    involves_test = leaked[leaked['splits'].apply(lambda s: 'test' in s)]
    print(f"  {len(involves_test)} families "
          f"({100 * len(involves_test) / max(n_leaked, 1):.2f}% of leaked families)")

print("\n`leaked` DataFrame is available for direct inspection, "
      "e.g. leaked.head(20)")

Using train/val/test already in memory.
Split lookup built: 12620 unique texts across train=10096, val=1262, test=1262

Re-melting abstracts with family_id...
Re-melting social media with family_id...
Total re-melted rows: 58530

Matched 18093 of 58530 re-melted rows against your saved corpus (30.9%). The rest were dropped by your existing length filter, dedup, or 50/50 balancing step, which is expected.

FAMILY LEAKAGE REPORT
Families with at least one row in the saved corpus : 11706
Families split across more than one partition       : 1529 (13.06%)
Total rows belonging to a leaked family              : 4682 out of 18093 matched rows (25.88%)

Breakdown by which splits are involved:
splits
['test', 'train']           687
['train', 'val']            684
['test', 'train', 'val']    114
['test', 'val']              44

How many leaked families involve the test set specifically (the number that matters most, since this is what inflates reported accuracy):
  845 families (55.26% of leaked

---

### What you have now

A single corpus `arabic_ai_detection_corpus.parquet` with columns:

| column | meaning |
|---|---|
| `text` | the cleaned, normalized Arabic text |
| `label` | `human` or `ai` |
| `domain` | `research` or `social_media` |
| `source` | `ASJP` or `BRAD+HARD` |
| `llm` | `human` for HW rows; one of `allam · jais · llama · openai` for AI rows |
| `method` | abstract generation method (`by_polishing` / `from_title` / `from_title_and_content`) |
| `n_words` | token count after normalization |
| `split` | `train` / `val` / `test` |

### Suggested next steps

1. **Train your Aero-CamelBERT-X / Aero-BiLSTM-X** on `train.parquet`, monitoring on `val.parquet`.
2. **Hold `test.parquet` out** until final reporting.
3. For **multi-class generator attribution** experiments later, you can re-use the same data
   with `llm` as the target instead of `label`.
4. To **stress-test cross-domain robustness**, train on `domain == 'research'` only and evaluate on `domain == 'social_media'` (and vice versa).
